# HW3: Multi-Agent Orchestration, State, and Evaluation

**Alumni RAG Agent — CMU Africa Alumni Tracking System**

This notebook demonstrates the HW3 enhancements:
1. **Persistent Memory** — Cross-session continuity via MongoDB
2. **Role Separation** — PlannerNode → ExecutorNode → CriticNode
3. **Evaluation Framework** — 4 metrics across 5 test cases
4. **Adaptive Control** — Closed-loop behavior with feedback-driven adaptation

## 1. Environment Setup

In [ ]:
import os
import sys
import json
import logging

# Add parent directory to path
sys.path.insert(0, os.path.dirname(os.getcwd()))

from dotenv import load_dotenv
load_dotenv()

# Enable LangSmith tracing
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "alumni-rag-agent-hw3"

# Configure logging to show agent internals
logging.basicConfig(level=logging.INFO, format='[%(asctime)s] %(levelname)s: %(message)s')

# Verify environment
print("Environment Check:")
print(f"  MongoDB URI: {'Set' if os.environ.get('MONGODB_URI') else 'MISSING'}")
print(f"  OpenAI API Key: {'Set' if os.environ.get('OPENAI_API_KEY') else 'MISSING'}")
print(f"  LangSmith API Key: {'Set' if os.environ.get('LANGCHAIN_API_KEY') else 'MISSING'}")
print(f"  Tavily API Key: {'Set' if os.environ.get('TAVILY_API_KEY') else 'MISSING'}")

## 2. Initialize Agent (with Persistent Memory + Role Separation)

In [ ]:
from src.agent import AlumniAgent, SAMPLE_ALUMNI

# Initialize — now includes PersistentMemory + role-separated ReActAgent
agent = AlumniAgent()

# ANNOTATION: The init logs should show:
#   ✓ Retrieval module initialized
#   ✓ Persistent memory initialized       ← NEW in HW3
#   ✓ Tools initialized
#   ✓ Verification module initialized
#   ✓ ReAct agent initialized with role separation + persistent memory  ← NEW in HW3

print(f"\nSample alumni data available: {len(SAMPLE_ALUMNI)} profiles")
for a in SAMPLE_ALUMNI:
    print(f"  - {a['name']} ({a['program']} {a['graduation_year']})")

## 3. Ingest Alumni Data

In [ ]:
# Ingest sample profiles into the vector store
chunk_count = agent.ingest_alumni(SAMPLE_ALUMNI)
print(f"Ingested {chunk_count} document chunks from {len(SAMPLE_ALUMNI)} profiles")

---

## 4. Annotated Execution Trace — Session 1

This demonstrates the **Planner → Executor → Critic** loop with memory write.

### Key things to watch in the logs:
- `[ORCHESTRATOR]` — Main loop coordination
- `[PLANNER]` — LLM reasoning and tool selection
- `[EXECUTOR]` — Prerequisite validation and tool execution
- `[CRITIC]` — Groundedness evaluation and adaptive decisions
- `[MEMORY WRITE]` — Session saved to MongoDB

In [ ]:
# SESSION 1: Query about alumni
result1 = agent.run("Tell me about alumni working in fintech")

print("\n" + "="*60)
print("SESSION 1 RESPONSE:")
print("="*60)
print(result1["response"])
print(f"\nSession ID: {result1.get('session_id', 'N/A')}")
print(f"Iterations used: {result1.get('iterations', 'N/A')}")
print(f"Actions taken: {result1.get('actions', [])}")

### 4a. Annotated Trace — Session 1

Each trace step shows which **role** performed it and what happened.

In [ ]:
# Pretty-print the annotated trace
print("ANNOTATED EXECUTION TRACE — Session 1")
print("=" * 60)

ANNOTATIONS = {
    "MEMORY_READ": "← Memory loaded from MongoDB (cross-session context)",
    "RETRIEVE": "← Initial document retrieval from vector store",
    "PLAN": "← Planner decides next action via LLM + bound tools",
    "EXECUTE": "← Executor validates prereqs and runs the tool",
    "CRITIQUE": "← Critic evaluates groundedness and recommends next step",
    "ADAPT": "← Adaptive control: behavior changed based on feedback",
    "VERIFY": "← Final groundedness verification of response",
    "MEMORY_WRITE": "← Session saved to MongoDB for future reference",
}

for i, step in enumerate(result1["trace"], 1):
    phase = step.get("phase", "UNKNOWN")
    role = step.get("role", "")
    annotation = ANNOTATIONS.get(phase, "")
    
    print(f"\n--- Step {i}: [{phase}] (Role: {role}) {annotation}")
    
    # Show relevant details per phase
    if phase == "MEMORY_READ":
        print(f"    Sessions loaded: {step.get('sessions_loaded', 0)}")
        print(f"    Task history matches: {step.get('task_history_matches', 0)}")
    elif phase == "RETRIEVE":
        print(f"    Result: {step.get('result', 'N/A')}")
    elif phase == "PLAN":
        print(f"    Action: {step.get('action', 'N/A')}")
        print(f"    Tool: {step.get('tool', 'None')}")
        print(f"    Reasoning: {step.get('reasoning', '')[:150]}...")
    elif phase == "EXECUTE":
        print(f"    Success: {step.get('success', 'N/A')}")
        print(f"    Tool used: {step.get('tool_used', 'None')}")
        if step.get('output'):
            print(f"    Output: {step.get('output', '')[:100]}")
        if step.get('error'):
            print(f"    Error: {step.get('error', '')}")
    elif phase == "CRITIQUE":
        print(f"    Continue: {step.get('should_continue', 'N/A')}")
        print(f"    Confidence: {step.get('confidence', 'N/A')}")
        print(f"    Groundedness: {step.get('groundedness', 'N/A')}")
        print(f"    Recommendation: {step.get('recommendation', 'N/A')}")
        print(f"    Feedback: {step.get('feedback', '')[:100]}")
    elif phase == "ADAPT":
        print(f"    Action: {step.get('action', 'N/A')}")
        print(f"    Result: {step.get('result', step.get('reason', 'N/A'))}")
    elif phase == "VERIFY":
        print(f"    Score: {step.get('score', 'N/A')}")
        print(f"    Confidence: {step.get('confidence', 'N/A')}")
    elif phase == "MEMORY_WRITE":
        print(f"    Session ID: {step.get('session_id', 'N/A')}")
        print(f"    Data saved: {step.get('data_saved', 'N/A')}")

---

## 5. Cross-Session Memory Demonstration

**HW3 Requirement:** "A later session using information stored in an earlier session."

We run a second query. The memory read should show Session 1's data being loaded.

In [ ]:
# SESSION 2: Follow-up query — memory should contain Session 1 context
result2 = agent.run("Send a check-in email to the fintech alumni we discussed")

print("\n" + "="*60)
print("SESSION 2 RESPONSE:")
print("="*60)
print(result2["response"])
print(f"\nSession ID: {result2.get('session_id', 'N/A')}")
print(f"Iterations used: {result2.get('iterations', 'N/A')}")

In [ ]:
# Show memory read from Session 2 trace
print("MEMORY READ EVIDENCE — Session 2")
print("=" * 60)

for step in result2["trace"]:
    if step.get("phase") == "MEMORY_READ":
        print(f"Sessions loaded from MongoDB: {step.get('sessions_loaded', 0)}")
        print(f"Task history matches: {step.get('task_history_matches', 0)}")
        print("\n→ Session 2 loaded Session 1's data from persistent memory.")
        print("→ This demonstrates cross-session continuity.")
        break

# Also show that Session 1 was saved
print("\nMemory contents (recent sessions):")
recent = agent.memory.get_recent_sessions(limit=5)
for s in recent:
    print(f"  [{s.get('session_id', 'N/A')}] Query: {s.get('query', '')[:60]} | "
          f"Tools: {s.get('tools_used', [])}")

---

## 6. Evaluation Framework — 5 Test Cases

Running all 5 structured test cases and computing 4 metrics:
1. **Groundedness Score** — Is the response supported by retrieved data?
2. **Tool Selection Accuracy** — Did the agent pick the correct tool?
3. **Iteration Efficiency** — How quickly did the agent converge?
4. **Task Completion** — Did the agent produce a valid response?

In [ ]:
from src.evaluation import EvaluationFramework, TEST_CASES

evaluator = EvaluationFramework(max_iterations=5)

print(f"Running {len(TEST_CASES)} test cases...")
print("=" * 60)

for tc in TEST_CASES:
    print(f"\n--- Test {tc.id}: {tc.name} ---")
    print(f"    Query: {tc.query}")
    print(f"    Expected tool: {tc.expected_tool or 'None (retrieval only)'}")
    if tc.is_failure_case:
        print(f"    ⚠️  INTENTIONAL FAILURE CASE")
    
    try:
        # Run the agent
        result = agent.run(tc.query)
        
        # Evaluate the run
        eval_result = evaluator.evaluate_run(tc, result)
        
        print(f"    Result: {'PASS' if eval_result.passed else 'FAIL'}")
        print(f"    Groundedness: {eval_result.groundedness_score:.2f}")
        print(f"    Tool accuracy: {eval_result.tool_selection_accuracy:.1f}")
        print(f"    Efficiency: {eval_result.iteration_efficiency:.2f}")
        print(f"    Completion: {eval_result.task_completion:.1f}")
    except Exception as e:
        print(f"    ERROR: {e}")

### 6a. Evaluation Results Table

In [ ]:
# Display results as a formatted table
import pandas as pd

rows = []
for r in evaluator.results:
    rows.append({
        "Test": f"{r.test_id}. {r.test_name}",
        "Groundedness": f"{r.groundedness_score:.2f}",
        "Tool Accuracy": f"{r.tool_selection_accuracy:.1f}",
        "Efficiency": f"{r.iteration_efficiency:.2f}",
        "Completion": f"{r.task_completion:.1f}",
        "Iterations": r.iterations_used,
        "Pass/Fail": "PASS" if r.passed else "FAIL",
        "Notes": r.notes[:40]
    })

df = pd.DataFrame(rows)
print("\nEVALUATION RESULTS")
print("=" * 100)
print(df.to_string(index=False))

### 6b. Summary Statistics

In [ ]:
stats = evaluator.get_summary_stats()

print("AGGREGATE STATISTICS")
print("=" * 40)
print(f"Total tests:        {stats.get('total_tests', 0)}")
print(f"Pass rate:          {stats.get('pass_rate', 0):.0%}")
print(f"Avg groundedness:   {stats.get('avg_groundedness', 0):.2f}")
print(f"Avg tool accuracy:  {stats.get('avg_tool_accuracy', 0):.2f}")
print(f"Avg efficiency:     {stats.get('avg_efficiency', 0):.2f}")
print(f"Avg completion:     {stats.get('avg_completion', 0):.2f}")
print(f"Failure cases:      {stats.get('failure_cases', 0)}")

### 6c. Failure Case Deep Dive — Test 5

In [ ]:
# Generate failure analysis report
failure_analysis = evaluator.get_failure_analysis()
print(failure_analysis)

---

## 7. Adaptive Control Demonstration

The trace should show the Critic triggering adaptive behavior:
- `RE_RETRIEVE` when groundedness is low
- `RE_PLAN` when a tool is blocked
- `ESCALATE` at max iterations
- `CLARIFY` on low confidence

In [ ]:
# Demonstrate adaptive control with the failure case trace
print("ADAPTIVE CONTROL EVIDENCE")
print("=" * 60)

# Run the failure case and inspect adaptive steps
failure_result = agent.run("Email someone in tech")

print("\nTrace steps showing adaptive behavior:")
for step in failure_result["trace"]:
    phase = step.get("phase", "")
    if phase in ("CRITIQUE", "ADAPT"):
        role = step.get("role", "")
        print(f"\n  [{phase}] Role: {role}")
        if phase == "CRITIQUE":
            print(f"    Continue: {step.get('should_continue')}")
            print(f"    Recommendation: {step.get('recommendation')}")
            print(f"    Feedback: {step.get('feedback', '')[:120]}")
        elif phase == "ADAPT":
            print(f"    Action: {step.get('action')}")
            print(f"    Result: {step.get('result', step.get('reason', ''))}")

print("\n→ The Observe → Reason → Decide → Act → Evaluate → Update loop")
print("  is visible as: RETRIEVE → PLAN → EXECUTE → CRITIQUE → (ADAPT) → repeat")

---

## 8. Export Trace for Report

In [ ]:
# Export trace as JSON for the evaluation report
import json

trace_export = {
    "session_1": {
        "query": "Tell me about alumni working in fintech",
        "session_id": result1.get("session_id"),
        "iterations": result1.get("iterations"),
        "trace": result1.get("trace", []),
        "verification_score": result1["verification"].score,
    },
    "session_2": {
        "query": "Send a check-in email to the fintech alumni we discussed",
        "session_id": result2.get("session_id"),
        "iterations": result2.get("iterations"),
        "trace": result2.get("trace", []),
        "verification_score": result2["verification"].score,
    },
    "evaluation": {
        "results_table": evaluator.get_results_table(),
        "summary_stats": evaluator.get_summary_stats(),
        "failure_analysis": evaluator.get_failure_analysis()
    }
}

# Save to file
with open("notebooks/hw3_trace_export.json", "w") as f:
    json.dump(trace_export, f, indent=2, default=str)

print("Trace exported to notebooks/hw3_trace_export.json")
print(f"\nLangSmith Project: {os.environ.get('LANGCHAIN_PROJECT', 'Not set')}")
print("View traces at: https://smith.langchain.com/")